# プレイヤーの位置情報の平均を描画する

## パス関係の定義

In [1]:
import sys
from pathlib import Path

# src ディレクトリのパスを取得して追加
src_dir = str(Path().resolve().parents[1])
if src_dir not in sys.path:
    sys.path.append(src_dir)

## データの取り出し

描画時は各マッチ毎，プレイヤー毎の平均値を記録することになるので`match_type -> player_name -> game_name`の順に整えつつデータの平均を取る

In [2]:
from analyzers import datamanager

data_manager = datamanager.DataManager("../processed_data")

trajectories = data_manager.get_all_trajectories()

# 描画用に全データの最大値を求める．
from analyzers import utils, calculator

all_player_names = data_manager.get_all_player_names()

draw_data_ave = {}
draw_data_std = {}

for player_name in all_player_names:
    player_trajectories = data_manager.get_data_by_player(player_name)

    # 特徴量を辞書に記録する
    averages = calculator.dict_average(player_trajectories)
    std_devs = calculator.dict_std_dev(player_trajectories)
    
    for match_type, games in averages.items():
        draw_data_ave.setdefault(match_type, {})[player_name] = games

    for match_type, games in std_devs.items():
        draw_data_std.setdefault(match_type, {})[player_name] = games

Match Type:  odict_keys(['AB', 'BD', 'CB', 'G1', 'G2', 'G3', 'G4', 'after', 'before'])


## 描画処理

drawerでは1データずつ記録していくのでループ処理で描画を繰り返す

### 前処理

色の設定やグラフの描画範囲を設定する

In [3]:
import numpy as np
import matplotlib.pyplot as plt

from analyzers import utils
from analyzers.features import drawer

colors_dict = {
    'O1red': 'red',
    'O2blue': 'blue',
    'O3pink': 'pink',
    'D1black': 'black',
    'D2orange': 'orange',
    'D3yellow': 'yellow'
}

max_ave = utils.round_up(calculator.max_recursive(draw_data_ave))
max_std = utils.round_up(calculator.max_recursive(draw_data_std))

### 平均値の描画処理

In [4]:
for match_type, player_data in draw_data_ave.items():
    fig, ax = plt.subplots(figsize=(12, 6))
    for idx, (player_name, games) in enumerate(player_data.items()):
        color = colors_dict[player_name]

        drawer.plot_feature(
            data = np.array(list(games.values())),
            ax=ax,
            position=idx,
            bar_kwargs={
                'color': color,
                'edgecolor': 'black',
                'linewidth': 1.2,
                'width': 0.6,
                'alpha': 0.5,
            },
            scatter_kwargs={
                'color': color,
                'edgecolor': 'black',
                'label': player_name,
                'alpha': 0.5,
                's': 20
            }
        )
    ax.set_xticks(range(len(player_data)))
    ax.set_xticklabels(all_player_names)
    ax.set_title(f'Average voronoi area in match_type {match_type}')
    ax.set_ylim(0, max_ave)
    ax.set_xlabel('Player')
    ax.set_ylabel('Average Voronoi Area (cm^2)')

    # 画像の保存処理    
    results_dir = Path('./results')
    results_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(f'./results/average_{match_type}.png', bbox_inches='tight')
    # plt.show()
    plt.close()

### 標準偏差の描画処理

In [5]:
for match_type, player_data in draw_data_std.items():
    fig, ax = plt.subplots(figsize=(12, 6))
    for idx, (player_name, games) in enumerate(player_data.items()):
        color = colors_dict[player_name]

        drawer.plot_feature(
            data = np.array(list(games.values())),
            ax=ax,
            position=idx,
            bar_kwargs={
                'color': color,
                'edgecolor': 'black',
                'linewidth': 1.2,
                'width': 0.6,
                'alpha': 0.5,
            },
            scatter_kwargs={
                'color': color,
                'edgecolor': 'black',
                'label': player_name,
                'alpha': 0.5,
                's': 20
            }
        )
    ax.set_xticks(range(len(player_data)))
    ax.set_xticklabels(all_player_names)
    ax.set_title(f'Standard deviation voronoi area in match_type {match_type}')
    ax.set_ylim(0, max_std)
    ax.set_xlabel('Player')
    ax.set_ylabel('Standard deviation Voronoi Area (cm^2)')

    # 画像の保存処理    
    results_dir = Path('./results')
    results_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(f'./results/std_devs_{match_type}.png', bbox_inches='tight')
    # plt.show()
    plt.close()    
